In [2]:
# Instalación de PySpark
!pip install pyspark

# Instalación de Java (Spark lo necesita para correr)
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

¡Spark está listo y funcionando!
Versión de Spark: 4.0.2


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# 1. INICIALIZACIÓN (Corregido sapark -> spark)
spark = SparkSession.builder \
    .appName("RetailMax_Ecommerce_Pipeline") \
    .getOrCreate()

# 2. CARGA DE DATOS (Asegúrate que el archivo esté en la raíz o ajusta la ruta)
# Si el archivo está fuera de la carpeta Data, usa "Ecommerce.csv"
try:
    df = spark.read.csv("Ecommerce.csv", header=True, inferSchema=True)
except:
    df = spark.read.csv("Data/Ecommerce.csv", header=True, inferSchema=True)

# Limpieza de nombres de columnas (Quitar puntos y espacios para evitar errores en SQL/ML)
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.replace(" ", "_").replace(".", ""))

# 3. PROCESAMIENTO
df_processed = df.withColumn("High_Spender", F.when(F.col("Yearly_Amount_Spent") > 500, 1).otherwise(0))
df_processed.createOrReplaceTempView("ecommerce_table")

# Consulta SQL con nombres corregidos
df_metrics = spark.sql("""
    SELECT High_Spender,
           AVG(Time_on_App) as avg_time_app,
           AVG(Length_of_Membership) as avg_membership
    FROM ecommerce_table
    GROUP BY High_Spender
""")
df_metrics.show()

# 4. PIPELINE DE MACHINE LEARNING
feature_cols = ["Avg_Session_Length", "Time_on_App", "Time_on_Website", "Length_of_Membership"]
df_ml = df_processed.dropna(subset=feature_cols)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="unscaled_features")
scaler = StandardScaler(inputCol="unscaled_features", outputCol="features")
lr = LogisticRegression(labelCol="High_Spender", featuresCol="features")

pipeline = Pipeline(stages=[assembler, scaler, lr])

# Entrenamiento
train_data, test_data = df_ml.randomSplit([0.7, 0.3], seed=42)
model = pipeline.fit(train_data)
predictions = model.transform(test_data)

# 5. EVALUACIÓN
evaluator = BinaryClassificationEvaluator(labelCol="High_Spender")
auc = evaluator.evaluate(predictions)
print(f"\n--- RESULTADO FINAL ---")
print(f"AUC (Precisión del modelo): {auc:.4f}")

+------------+------------------+------------------+
|High_Spender|      avg_time_app|    avg_membership|
+------------+------------------+------------------+
|           1|12.395383117802679|4.1768994265721675|
|           0| 15.96272097351788| 85.25428184397303|
+------------+------------------+------------------+


--- RESULTADO FINAL ---
AUC (Precisión del modelo): 0.7989


In [5]:
!readlink -f Ecommerce.csv

/content/Ecommerce.csv


In [6]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Configuración del entorno (Asegúrate de haber corrido la instalación antes)
spark = SparkSession.builder.appName("RetailMax_Final").getOrCreate()

# 2. Definir la ruta absoluta
# En Colab, los archivos subidos manualmente suelen quedar en /content/
file_path = "/content/Ecommerce.csv"

if os.path.exists(file_path):
    print(f"Archivo encontrado en: {file_path}")

    # Carga del archivo usando la ruta absoluta
    df = spark.read.csv(file_path, header=True, inferSchema=True)

    # Limpiar nombres de columnas (importante para evitar errores posteriores)
    for col_name in df.columns:
        new_name = col_name.replace(" ", "_").replace(".", "")
        df = df.withColumnRenamed(col_name, new_name)

    # Mostrar que cargó correctamente
    print("Primeros 5 registros cargados:")
    df.show(5)

    # Proceder con el procesamiento de RetailMax
    df_processed = df.withColumn("High_Spender", F.when(F.col("Yearly_Amount_Spent") > 500, 1).otherwise(0))
    df_processed.createOrReplaceTempView("ecommerce_table")

    print("Métricas iniciales de gasto:")
    spark.sql("SELECT High_Spender, COUNT(*) as cantidad FROM ecommerce_table GROUP BY High_Spender").show()

else:
    print(f"ERROR: No se encontró el archivo en {file_path}")
    print("Archivos disponibles en la carpeta actual:")
    print(os.listdir('/content/'))

ERROR: No se encontró el archivo en /content/Ecommerce.csv
Archivos disponibles en la carpeta actual:
['.config', 'sample_data']
